In [1]:
import torch
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))
    print("Versão CUDA do PyTorch:", torch.version.cuda)

CUDA disponível: True
Nome da GPU: NVIDIA GeForce GTX 1650
Versão CUDA do PyTorch: 12.1


In [2]:
dimension = 10

# Arquitetura 
As entradas foram reduzidas para 10x10. 
É uma rede simples, com uma camada fully connected. 


## Carregar o dataset

In [3]:
from torchvision import datasets, transforms
import os

transform = transforms.Compose([
    transforms.Grayscale(),     
    transforms.Resize((75, 100)),
    transforms.ToTensor()
])

!ls CNN_letter_Dataset/test

test_data  = datasets.ImageFolder(root="CNN_letter_Dataset/test", transform=transform)
train_data  = datasets.ImageFolder(root="CNN_letter_Dataset/train", transform=transform)

# --- DataLoaders ---
train_loader = torch.utils.data.DataLoader(train_data, batch_size=10, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_data, batch_size=10, shuffle=False)

# --- Informações básicas ---
print(f"Total de imagens de treino: {len(train_data)}")
print(f"Total de imagens de teste:  {len(test_data)}")

0  2  4  6  8  A  C  E	G  I  K  M  P  R  T  V	X  Z
1  3  5  7  9  B  D  F	H  J  L  N  Q  S  U  W	Y
Total de imagens de treino: 28400
Total de imagens de teste:  7100


### Normalizar o dataset

In [4]:
import torch.nn.functional as F

interpolate = True
important = False
def interpolate_image(img_tensor, new_dim=dimension):
    """
    Recebe uma imagem [C,H,W] ou [H,W] e retorna vetor [num_top_pixels]
    contendo apenas os pixels mais importantes no novo espaço.
    """
    # Remove canal se necessário
    img = img_tensor.squeeze()  # [H, W]
    img_resized = F.interpolate(
        img.unsqueeze(0).unsqueeze(0),  # adiciona batch e canal
        size=(new_dim, new_dim),
        mode='bilinear',
        align_corners=False
    ).squeeze()  # remove batch e canal

    return img_resized.flatten()  

In [5]:
from torch.utils.data import TensorDataset, DataLoader
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reduz conjunto de treino
#X_train_reduced = torch.stack([reduce_image(img, top_pixel_coordinates_int) for img, _ in train_data])
X_train_reduced = torch.stack([interpolate_image(img) for img, _ in train_data])
y_train = torch.tensor([label for _, label in train_data])


# Reduz conjunto de teste
X_test_reduced = torch.stack([interpolate_image(img) for img, _ in test_data])
y_test = torch.tensor([label for _, label in test_data])

print(f"Novo shape de treino: {X_train_reduced.shape}")
print(f"Novo shape de teste:  {X_test_reduced.shape}")

# DataLoaders
batch_size = 10
train_loader_reduced = DataLoader(TensorDataset(X_train_reduced, y_train), batch_size=batch_size, shuffle=True)
test_loader_reduced = DataLoader(TensorDataset(X_test_reduced, y_test), batch_size=batch_size)


Novo shape de treino: torch.Size([28400, 100])
Novo shape de teste:  torch.Size([7100, 100])


## Modelo

In [6]:
import torch.nn as nn

num_pixels = 10 * 10  
n_classes = len(train_data.classes)    

model_small = nn.Sequential(
    nn.Linear(num_pixels, n_classes)
)


## Treinamento

In [7]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = "cpu"
model_small.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_small.parameters(), lr=0.1)

epochs = 5
losses_interpolate = []

for epoch in range(epochs):
    print(f'Epoch: {epoch}')
    model_small.train()
    total_loss = 0
    
    for X_batch, y_batch in tqdm(train_loader_reduced):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        output = model_small(X_batch)
        loss = criterion(output, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    

    losses_interpolate.append(total_loss / len(train_loader_reduced))
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader_reduced):.4f}")



Epoch: 0


100%|███████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:36<00:00, 77.23it/s]


Epoch 1, Loss: 1.8692
Epoch: 1


100%|██████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:06<00:00, 462.58it/s]


Epoch 2, Loss: 0.9450
Epoch: 2


...

Epoch 3, Loss: 0.7313
Epoch: 3


100%|██████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:05<00:00, 495.98it/s]


Epoch 4, Loss: 0.6256
Epoch: 4


100%|██████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:07<00:00, 400.26it/s]


Epoch 5, Loss: 0.5593


In [8]:
print(losses_interpolate)

[1.869175703802579, 0.9449967129444572, 0.7313486836011142, 0.6255757253617048, 0.5593446778500794]


## Teste

In [9]:
from torch.utils.data import TensorDataset, DataLoader
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Lista de imagens reduzidas usando as coordenadas corretas
X_test_reduced = torch.stack([interpolate_image(img) for img, _ in test_data])
y_test = torch.tensor([label for _, label in test_data])

print(f"Shape do dataset reduzido: {X_test_reduced.shape}")  # deve ser (N_test, num_top_pixels)

# DataLoader para o modelo reduzido
test_loader_reduced = DataLoader(
    TensorDataset(X_test_reduced, y_test),
    batch_size=10,  # ou outro batch_size que quiser
    shuffle=False
)

Shape do dataset reduzido: torch.Size([7100, 100])


In [10]:
def calc_loss_reduced(model, criterion, loader):
    """
    Calcula a loss média de um modelo MLP que recebe entradas já achatadas.
    loader: DataLoader do dataset reduzido (ex: test_loader_reduced)
    """
    batch_losses = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        for img_batch, label_batch in loader:
            img_batch = img_batch.to(device)
            label_batch = label_batch.to(device)

            output = model(img_batch)
            loss = criterion(output, label_batch).item()
            batch_losses.append(loss)

    mean_loss = sum(batch_losses) / len(batch_losses)
    return batch_losses, mean_loss


In [11]:
batch_losses_interpolate, mean_loss_interpolate = calc_loss_reduced(model_small, criterion, test_loader_reduced)
print(f"Loss média: {mean_loss_interpolate:.4f}")
print(f"Loss por batch (primeiros 5): {batch_losses_interpolate[:10]}")


Loss média: 0.5455
Loss por batch (primeiros 5): [0.7788294553756714, 1.2015488147735596, 1.2879167795181274, 0.40384984016418457, 2.1629693508148193, 0.8164388537406921, 0.763969361782074, 1.12766695022583, 0.63469398021698, 0.8778292536735535]


In [12]:
def calc_accuracy_reduced(model, loader, n_classes=None):
    """
    Calcula acurácia geral e por classe de um modelo MLP.
    loader: DataLoader do dataset reduzido.
    n_classes: número de classes (se None, inferido do loader).
    """
    if n_classes is None:
        n_classes = len(loader.dataset.tensors[1].unique())
    
    correct_total = 0
    total_total = 0
    correct_per_class = {i: 0 for i in range(n_classes)}
    total_per_class = {i: 0 for i in range(n_classes)}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        for img_batch, label_batch in loader:
            img_batch = img_batch.to(device)
            label_batch = label_batch.to(device)

            output = model(img_batch)
            preds = output.argmax(dim=1)

            correct_total += (preds == label_batch).sum().item()
            total_total += len(label_batch)

            for i in range(len(label_batch)):
                label = label_batch[i].item()
                total_per_class[label] += 1
                if preds[i] == label_batch[i]:
                    correct_per_class[label] += 1

    class_acc = {cls: correct_per_class[cls]/total_per_class[cls] 
                 if total_per_class[cls] > 0 else 0
                 for cls in range(n_classes)}

    overall_acc = correct_total / total_total
    return overall_acc, class_acc


In [13]:
overall_acc_interpolate, class_acc_interpolate = calc_accuracy_reduced(model_small, test_loader_reduced)
print(f"Acurácia geral: {overall_acc_interpolate*100:.2f}%")
for cls, acc in class_acc_interpolate.items():
    print(f"Classe {cls}: {acc*100:.2f}%")

Acurácia geral: 88.52%
Classe 0: 80.10%
Classe 1: 91.75%
Classe 2: 92.23%
Classe 3: 91.26%
Classe 4: 96.12%
Classe 5: 95.15%
Classe 6: 83.50%
Classe 7: 82.04%
Classe 8: 72.82%
Classe 9: 91.26%
Classe 10: 87.62%
Classe 11: 76.70%
Classe 12: 94.61%
Classe 13: 97.03%
Classe 14: 95.05%
Classe 15: 94.12%
Classe 16: 93.63%
Classe 17: 91.67%
Classe 18: 46.04%
Classe 19: 97.57%
Classe 20: 97.03%
Classe 21: 93.56%
Classe 22: 93.63%
Classe 23: 92.16%
Classe 24: 94.06%
Classe 25: 75.25%
Classe 26: 93.14%
Classe 27: 90.20%
Classe 28: 91.67%
Classe 29: 89.11%
Classe 30: 83.50%
Classe 31: 91.09%
Classe 32: 91.09%
Classe 33: 78.22%
Classe 34: 95.68%


In [2]:
from MAxPy import maxpy


In [1]:
import pytimeloop.timeloopfe.v4 as tl
import os
import shutil # Importado para limpar a pasta 'outputs'

# --- Adicionado: Limpeza da pasta de saída ---
# Isso evita erros se a pasta 'outputs' já existir
output_dir = f"{os.curdir}/outputs"
if os.path.exists(output_dir):
    print(f"Limpando pasta de saída antiga: {output_dir}")
    shutil.rmtree(output_dir)
# --- Fim da adição ---

# 1. Defina o caminho para o arquivo "agregador"
TOP_PATH = f"{os.curdir}/top.yaml.jinja"

# 2. Carregue a especificação
spec = tl.Specification.from_yaml_files(TOP_PATH)

# 3. Modifique para seu DSE
#    Vamos alterar a profundidade do buffer
#    ALTERAÇÃO 1: O nome agora é "Buffer" (maiúsculo)
spec.architecture.find("Buffer").attributes.depth = 4096 

# 4. Rode o mapper
print("Rodando o Timeloop Mapper...")


try:
    result = tl.call_mapper(spec, output_dir=output_dir)

    # 5. Imprima os resultados
    print("\nSimulação concluída!")
    stats = result.get_stats()
    print(f"Energia (pJ/MAC): {stats.per_compute('energy') * 1e12}")
    print(f"Ciclos: {stats.cycles}")

except Exception as e:
    print(f"\n--- A SIMULAÇÃO FALHOU ---")
    print(f"Erro: {e}")
    print("\nVerifique se os plugins do Accelergy para 'regfile' e 'intmac' estão instalados.")
# --- Fim da adição ---

Limpando pasta de saída antiga: ./outputs
Rodando o Timeloop Mapper...

--- A SIMULAÇÃO FALHOU ---
Erro: 

Timeloop mapper failed with return code 127. Please check the output files in ./outputs for more information. To debug, you can edit the file:
	./outputs/parsed-processed-input.yaml
and run 
	tl mapper ./outputs/parsed-processed-input.yaml
to see the error. If you're running the mapper and Timeloop can't find a vaild mapping, try setting 'diagnostics: true' in the mapper input specification.

Verifique se os plugins do Accelergy para 'regfile' e 'intmac' estão instalados.


In [45]:
import os
print("PATH USADO PELO PYTHON:\n", os.environ.get('PATH'))

PATH USADO PELO PYTHON:
 /home/juliana/venv_pytorch/bin:/home/juliana/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/usr/lib/wsl/lib:/mnt/c/WINDOWS/system32:/mnt/c/WINDOWS:/mnt/c/WINDOWS/System32/Wbem:/mnt/c/WINDOWS/System32/WindowsPowerShell/v1.0/:/mnt/c/WINDOWS/System32/OpenSSH/:/mnt/c/Program Files/Git/cmd:/mnt/c/Program Files/MATLAB/R2014a/runtime/win64:/mnt/c/Program Files/MATLAB/R2014a/bin:/mnt/c/Program Files/usbipd-win/:/mnt/c/Users/julia/AppData/Local/Programs/Python/Launcher/:/mnt/c/Users/julia/AppData/Local/Microsoft/WindowsApps:/mnt/c/Users/julia/AppData/Local/Programs/Microsoft VS Code/bin:/snap/bin:/home/juliana/.local/bin:/home/juliana/.local/bin:/home/juliana/.local/bin
